# Решения: практика буферов

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('orders_slim.csv')
df = pd.read_csv(
    CSV_PATH,
    parse_dates=['order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date'],
)

from collections import deque

In [ ]:
recent = deque(maxlen=7)
for oid in df['order_id']:
    recent.append(oid)
late_q, normal_q = deque(), deque()
for row in df[['order_id', 'is_late']].itertuples(index=False):
    if row.is_late == 1:
        late_q.append(row.order_id)
    else:
        normal_q.append(row.order_id)
processed = []
for _ in range(12):
    if late_q:
        processed.append(late_q.popleft())
    elif normal_q:
        processed.append(normal_q.popleft())
window = deque(maxlen=5)
means = []
for d in df['delivery_days'].tolist():
    window.append(float(d))
    if len(window) == 5:
        means.append(float(np.mean(window)))
LOAD_NOTE = (
    'Deque с maxlen позволяет держать скользящее окно без ручного удаления старых элементов. '
    'В пике late-очередь обрабатываем отдельно, чтобы видеть накопление критичных заказов.'
)
print(list(recent))
print(len(late_q), len(normal_q))
print(processed)
print(means[:5])
print(LOAD_NOTE)